# Анализ мультимодальных моделей / Multimodal Models Analysis

Ноутбук загружает обученные мультимодальные модели классификации эмоций речи и оценивает их
на тестовом наборе данных **Dusha** (`combine_balanced_test.lmdb`).

Для каждой модели выводятся:
- Сводные метрики (Accuracy, Balanced Accuracy, F1, MCC, ROC-AUC)
- Отчёт по классам (Precision / Recall / F1)
- Матрица ошибок (confusion matrix)

**Эмоции (4 класса):** `angry` · `sad` · `neutral` · `positive`

---
## Содержание

| # | Модель | Описание |
|---|--------|----------|
| 1 | [Late Fusion (CNN-BiLSTM + RuBERT)](#1-late-fusion-cnn-bilstm--rubert) | Soft-voting fusion двух PyTorch-моделей |
| 2 | [Late Fusion Baseline (SVM + TF-IDF LogReg)](#2-late-fusion-baseline-svm--tf-idf-logreg) | Soft-voting fusion двух sklearn-моделей |

In [1]:
import sys
from pathlib import Path

for _p in [Path.cwd()] + list(Path.cwd().parents):
    if _p.name == 'my_experiments':
        if str(_p.parent) not in sys.path:
            sys.path.append(str(_p.parent))
        break

from my_experiments.utils.config_utils import (
    PROJECT_ROOT, DATASET_PATH, TARGET_NAMES, EMO2LABEL,
)

import numpy as np
import torch
import warnings
warnings.filterwarnings('ignore')

AGGREGATED_DIR = DATASET_PATH / 'processed_dataset_090' / 'aggregated_dataset'

# ──────────────────────────────────────────────────────
# ВЫБОР ТЕСТОВОГО НАБОРА ДАННЫХ
# Раскомментируйте нужный путь:
# ──────────────────────────────────────────────────────
# TEST_LMDB = AGGREGATED_DIR / 'combine_balanced_test.lmdb'        # полный сбалансированный тест
# TEST_LMDB = AGGREGATED_DIR / 'combine_balanced_test_small.lmdb'  # малая версия
# TEST_LMDB = AGGREGATED_DIR / 'dusha_resd_test.lmdb'             # Dusha + RESD
# TEST_LMDB = AGGREGATED_DIR / 'crowd_test.lmdb'                  # только crowd
# TEST_LMDB = AGGREGATED_DIR / 'podcast_test.lmdb'                # только podcast
TEST_LMDB = AGGREGATED_DIR / 'combine_balanced_test.lmdb'

CHECKPOINTS_DIR = PROJECT_ROOT / 'my_experiments' / 'checkpoints'
AUDIO_CKPT = CHECKPOINTS_DIR / 'audio'
TEXT_CKPT = CHECKPOINTS_DIR / 'text'
MULTIMODAL_CKPT = CHECKPOINTS_DIR / 'multimodal'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Test dataset: {TEST_LMDB}')
print(f'Exists: {TEST_LMDB.exists()}')
print(f'Device: {DEVICE}')

Test dataset: /home/natlis/PycharmProjects/dusha_new/dusha/data_processing/dataset/processed_dataset_090/aggregated_dataset/combine_balanced_test.lmdb
Exists: True
Device: cpu


---
## Вспомогательные функции

In [2]:
import pickle
import lmdb
import re
from torch import nn
from torch.utils.data import DataLoader, Dataset

from my_experiments.utils.lmdb_utils import open_lmdb_readonly, get_lmdb_length, parse_label_to_index
from my_experiments.utils.model_io import load_sklearn_model, load_pytorch_model
from my_experiments.utils.metrics import print_eval_block, compute_classification_metrics


def extract_text(payload):
    for key in ('speaker_text', 'text', 'transcript', 'utterance'):
        if key in payload:
            text = str(payload[key]).strip().lower()
            text = re.sub(r'\s+', ' ', text)
            if text:
                return text
    return ''


def _to_fixed_vector(feat):
    arr = np.asarray(feat)
    arr = np.squeeze(arr)
    if arr.ndim == 1:
        return arr.astype(np.float32)
    return np.concatenate([arr.mean(axis=-1), arr.std(axis=-1)]).astype(np.float32)


def _normalize_waveform(raw_waveform):
    arr = np.asarray(raw_waveform)
    if arr.ndim != 1:
        arr = arr.reshape(-1)
    if arr.dtype == np.int16:
        arr = arr.astype(np.float32) / 32768.0
    else:
        arr = arr.astype(np.float32)
        arr = np.nan_to_num(arr, nan=0.0, posinf=1.0, neginf=-1.0)
        peak = np.max(np.abs(arr)) if arr.size > 0 else 1.0
        if peak > 1.0:
            arr = arr / peak
    return np.clip(np.nan_to_num(arr, nan=0.0, posinf=1.0, neginf=-1.0), -1.0, 1.0)

---
## 1. Late Fusion (CNN-BiLSTM + RuBERT)

**Тип:** Soft-voting fusion двух предобученных PyTorch моделей  
**Аудио:** `EmotionCNNBiLSTM` (из `CNN_BiLSTM_combine_balanced_train_model.pt`)  
**Текст:** `EmotionClassifier` (RuBERT, из `RuBERT_dusha_resd_train_model.pt`)  
**Fusion:** `alpha * P_audio + (1 - alpha) * P_text`, alpha подбирается grid search

In [3]:
from transformers import AutoTokenizer
from my_experiments.audio_models.CNN.CNN_BiLSTM import EmotionCNNBiLSTM
from my_experiments.text_models.transformers.RuBERT import EmotionClassifier

# ── Загрузка аудио-модели ──
audio_ckpt_path = AUDIO_CKPT / 'CNN_BiLSTM_combine_balanced_train_model.pt'
audio_checkpoint = torch.load(audio_ckpt_path, map_location=DEVICE, weights_only=False)
if 'model_state_dict' in audio_checkpoint:
    audio_state = audio_checkpoint['model_state_dict']
else:
    audio_state = audio_checkpoint

audio_model = EmotionCNNBiLSTM(n_classes=4)
audio_model.load_state_dict(audio_state, strict=False)
audio_model.to(DEVICE)
audio_model.eval()
print(f'Audio model loaded from {audio_ckpt_path}')

# ── Загрузка текстовой модели ──
text_ckpt_path = TEXT_CKPT / 'RuBERT_dusha_resd_train_model.pt'
text_checkpoint = torch.load(text_ckpt_path, map_location=DEVICE, weights_only=False)
text_model_params = text_checkpoint.get('model_params', {})

text_model = EmotionClassifier(
    model_name=text_model_params.get('backbone_name', 'DeepPavlov/rubert-base-cased'),
    num_classes=4,
    dropout=text_model_params.get('dropout', 0.1),
    classifier_hidden_size=text_model_params.get('classifier_hidden_size', None),
)
text_model.load_state_dict(text_checkpoint['model_state_dict'], strict=False)
text_model.to(DEVICE)
text_model.eval()

tokenizer_dir = TEXT_CKPT / 'RuBERT_dusha_resd_train_tokenizer'
if tokenizer_dir.exists():
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir)
else:
    tokenizer = AutoTokenizer.from_pretrained(
        text_model_params.get('backbone_name', 'DeepPavlov/rubert-base-cased')
    )
print(f'Text model loaded from {text_ckpt_path}')

max_len = int(text_model_params.get('max_len', 128))

Audio model loaded from /home/natlis/PycharmProjects/dusha_new/dusha/my_experiments/checkpoints/audio/CNN_BiLSTM_combine_balanced_train_model.pt


Some weights of the model checkpoint at DeepPavlov/rubert-base-cased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Text model loaded from /home/natlis/PycharmProjects/dusha_new/dusha/my_experiments/checkpoints/text/RuBERT_dusha_resd_train_model.pt


### Загрузка данных и получение предсказаний

In [4]:
env = open_lmdb_readonly(TEST_LMDB)
total = get_lmdb_length(env)

all_audio_probs = []
all_text_probs = []
all_labels = []

with env.begin() as txn:
    for idx in range(total):
        raw = txn.get(str(idx).encode('utf-8'))
        if raw is None:
            continue
        payload = pickle.loads(raw)
        if not isinstance(payload, dict):
            continue
        
        # Label
        label_raw = payload.get('y', payload.get('label', payload.get('emotion')))
        try:
            label = parse_label_to_index(label_raw)
        except ValueError:
            continue
        
        # Text
        text = extract_text(payload)
        if not text:
            continue
        
        # Audio features
        if 'x' not in payload:
            continue
        arr = np.asarray(payload['x'], dtype=np.float32)
        x = torch.from_numpy(arr)
        if x.ndim == 2:
            x = x.unsqueeze(0)
        x = x.unsqueeze(0).to(DEVICE)
        
        # Get audio prediction
        with torch.no_grad():
            logits_a = audio_model(x, torch.tensor([x.shape[-1]], device=DEVICE))
            probs_a = torch.softmax(logits_a, dim=1).cpu().numpy()[0]
        
        # Get text prediction
        enc = tokenizer(
            [text], padding='max_length', truncation=True,
            max_length=max_len, return_tensors='pt',
        )
        input_ids = enc['input_ids'].to(DEVICE)
        attn_mask = enc['attention_mask'].to(DEVICE)
        with torch.no_grad():
            logits_t = text_model(input_ids, attn_mask)
            probs_t = torch.softmax(logits_t, dim=1).cpu().numpy()[0]
        
        all_audio_probs.append(probs_a)
        all_text_probs.append(probs_t)
        all_labels.append(label)

env.close()

audio_probs = np.stack(all_audio_probs)
text_probs = np.stack(all_text_probs)
y_true = np.array(all_labels)
print(f'Processed {len(y_true)} samples')

Processed 6392 samples


### Grid search для alpha и финальная оценка

In [5]:
best_alpha = 0.0
best_f1 = 0.0
best_metrics = None

for alpha in np.arange(0.0, 1.05, 0.05):
    fused = alpha * audio_probs + (1.0 - alpha) * text_probs
    y_pred = np.argmax(fused, axis=1)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_alpha = alpha

fused_probs = best_alpha * audio_probs + (1.0 - best_alpha) * text_probs
y_pred_final = np.argmax(fused_probs, axis=1)
metrics_fusion = compute_classification_metrics(y_true, y_pred_final, fused_probs)
print(f'Best alpha: {best_alpha:.2f}')
print_eval_block('Late Fusion (CNN-BiLSTM + RuBERT) - Test Metrics', metrics_fusion, y_true, y_pred_final)

Best alpha: 0.50

Late Fusion (CNN-BiLSTM + RuBERT) - Test Metrics
            accuracy: 0.802253
   balanced_accuracy: 0.803927
     precision_macro: 0.798590
        recall_macro: 0.803927
            f1_macro: 0.798195
         f1_weighted: 0.798070
                 mcc: 0.733683
   roc_auc_ovr_macro: 0.943204

Classification report:
              precision    recall  f1-score   support

       angry     0.8050    0.9223    0.8596      1338
         sad     0.8232    0.8679    0.8450      2157
     neutral     0.7663    0.6205    0.6857      1681
    positive     0.7998    0.8051    0.8025      1216

    accuracy                         0.8023      6392
   macro avg     0.7986    0.8039    0.7982      6392
weighted avg     0.8000    0.8023    0.7981      6392

Confusion matrix:
[[1234   33   34   37]
 [  38 1872  188   59]
 [ 165  324 1043  149]
 [  96   45   96  979]]


---
## 2. Late Fusion Baseline (SVM + TF-IDF LogReg)

**Тип:** Soft-voting fusion двух sklearn-моделей  
**Аудио:** SVM (из `svm_combine_balanced_train_model.pkl`)  
**Текст:** TF-IDF + LogisticRegression (из `TF-IDF_LogReg_combine_balanced_train_model.pkl`)  
**Fusion:** `alpha * P_audio + (1 - alpha) * P_text`

In [6]:
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer

# ── Загрузка моделей ──
audio_model_svm = joblib.load(AUDIO_CKPT / 'svm_combine_balanced_train_model.pkl')
audio_scaler = joblib.load(AUDIO_CKPT / 'svm_combine_balanced_train_scaler.pkl')

text_model_tfidf = joblib.load(TEXT_CKPT / 'TF-IDF_LogReg_combine_balanced_train_model.pkl')
vectorizer = joblib.load(TEXT_CKPT / 'TF-IDF_LogReg_combine_balanced_train_vectorizer.pkl')

print('Models loaded successfully')
print(f'Audio SVM: {type(audio_model_svm).__name__}')
print(f'Text TF-IDF+LogReg: {type(text_model_tfidf).__name__}')

Models loaded successfully
Audio SVM: SVC
Text TF-IDF+LogReg: LogisticRegression


### Загрузка данных и получение предсказаний

In [7]:
env = open_lmdb_readonly(TEST_LMDB)
total = get_lmdb_length(env)

audio_feats_list = []
texts_list = []
labels_list = []

with env.begin() as txn:
    for idx in range(total):
        raw = txn.get(str(idx).encode('utf-8'))
        if raw is None:
            continue
        payload = pickle.loads(raw)
        if not isinstance(payload, dict):
            continue
        
        label_raw = payload.get('y', payload.get('label', payload.get('emotion')))
        try:
            label = parse_label_to_index(label_raw)
        except ValueError:
            continue
        
        text = extract_text(payload)
        if not text:
            continue
        
        if 'x' not in payload:
            continue
        audio_vec = _to_fixed_vector(np.asarray(payload['x'], dtype=np.float32))
        
        audio_feats_list.append(audio_vec)
        texts_list.append(text)
        labels_list.append(label)

env.close()

audio_feats = np.stack(audio_feats_list)
X_audio_scaled = audio_scaler.transform(audio_feats)
X_text_tfidf = vectorizer.transform(texts_list)
y_true_base = np.array(labels_list)
print(f'Processed {len(y_true_base)} samples')

Processed 6392 samples


### Функция proba и grid search для alpha

In [8]:
def get_proba(model, X):
    if hasattr(model, 'predict_proba'):
        probs = model.predict_proba(X)
    else:
        scores = model.decision_function(X)
        if scores.ndim == 1:
            scores = np.column_stack([-scores, scores])
        probs = np.exp(scores) / np.exp(scores).sum(axis=1, keepdims=True)
    return probs


def align_to_targets(probs, model_classes):
    aligned = np.zeros((probs.shape[0], len(TARGET_NAMES)))
    for i, cls_name in enumerate(TARGET_NAMES):
        cls_idx = EMO2LABEL[cls_name]
        if cls_idx in model_classes:
            col = np.where(model_classes == cls_idx)[0]
            if len(col) > 0:
                aligned[:, i] = probs[:, col[0]]
    return aligned


audio_probs_base = get_proba(audio_model_svm, X_audio_scaled)
audio_probs_base = align_to_targets(audio_probs_base, audio_model_svm.classes_)

text_probs_base = get_proba(text_model_tfidf, X_text_tfidf)
text_probs_base = align_to_targets(text_probs_base, text_model_tfidf.classes_)

best_alpha_base = 0.0
best_f1_base = 0.0
for alpha in np.arange(0.0, 1.05, 0.05):
    fused = alpha * audio_probs_base + (1.0 - alpha) * text_probs_base
    y_pred = np.argmax(fused, axis=1)
    f1 = f1_score(y_true_base, y_pred, average='macro', zero_division=0)
    if f1 > best_f1_base:
        best_f1_base = f1
        best_alpha_base = alpha

fused_probs_base = best_alpha_base * audio_probs_base + (1.0 - best_alpha_base) * text_probs_base
y_pred_base = np.argmax(fused_probs_base, axis=1)
metrics_fusion_base = compute_classification_metrics(y_true_base, y_pred_base, fused_probs_base)
print(f'Best alpha: {best_alpha_base:.2f}')
print_eval_block('Late Fusion Baseline (SVM + TF-IDF) - Test Metrics', metrics_fusion_base, y_true_base, y_pred_base)

Best alpha: 0.00

Late Fusion Baseline (SVM + TF-IDF) - Test Metrics
            accuracy: 0.209324
   balanced_accuracy: 0.250000
     precision_macro: 0.052331
        recall_macro: 0.250000
            f1_macro: 0.086546
         f1_weighted: 0.072465
                 mcc: 0.000000
   roc_auc_ovr_macro: nan

Classification report:
              precision    recall  f1-score   support

       angry     0.2093    1.0000    0.3462      1338
         sad     0.0000    0.0000    0.0000      2157
     neutral     0.0000    0.0000    0.0000      1681
    positive     0.0000    0.0000    0.0000      1216

    accuracy                         0.2093      6392
   macro avg     0.0523    0.2500    0.0865      6392
weighted avg     0.0438    0.2093    0.0725      6392

Confusion matrix:
[[1338    0    0    0]
 [2157    0    0    0]
 [1681    0    0    0]
 [1216    0    0    0]]
